In [4]:
# Naive Bayes Fraud Detection (Final Version)
# With Behavioural Feature Engineering + Threshold Tuning

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import KBinsDiscretizer, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import CategoricalNB

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)

# 1. Feature Engineering

def add_features(df_):
    df = pd.read_csv("Bank_Transaction_Fraud_Detection.csv" )

    # Base datetime features

    dt = pd.to_datetime(
        df["Transaction_Date"] + " " + df["Transaction_Time"],
        format="%d-%m-%Y %H:%M:%S",
        errors="coerce"
    )

    df["tx_datetime"]   = dt
    df["tx_hour"]       = dt.dt.hour
    df["tx_dayofweek"]  = dt.dt.dayofweek
    df["tx_month"]      = dt.dt.month
    df["is_weekend"]    = (df["tx_dayofweek"] >= 5).astype(int)
    df["is_night"]      = ((df["tx_hour"] < 6) | (df["tx_hour"] >= 22)).astype(int)

    df["hour_bin"] = pd.cut(
        df["tx_hour"],
        bins=[-1, 6, 12, 18, 24],
        labels=["Night", "Morning", "Afternoon", "Evening"]
    ).astype("object")


    # Amount features

    df["amount_log"] = np.log1p(df["Transaction_Amount"])
    df["is_round_amount"] = (
        ((df["Transaction_Amount"] % 10 == 0) |
         (df["Transaction_Amount"] % 100 == 0))
    ).astype(int)

    global_median = df["Transaction_Amount"].median()
    df["is_high_value"] = (df["Transaction_Amount"] > global_median).astype(int)

    df["amt_to_balance_ratio"] = df["Transaction_Amount"] / (df["Account_Balance"] + 1e-6)


    # Inflow / outflow flags

    inflow_types  = ["Credit"]
    outflow_types = ["Debit", "Withdrawal", "Transfer", "Bill Payment"]

    df["is_inflow"]  = df["Transaction_Type"].isin(inflow_types).astype(int)
    df["is_outflow"] = df["Transaction_Type"].isin(outflow_types).astype(int)


    # Inactivity (customer-level)

    df = df.sort_values(["Customer_ID", "tx_datetime"])
    df["prev_tx_time"] = df.groupby("Customer_ID")["tx_datetime"].shift(1)

    df["days_since_prev_tx"] = (
        (df["tx_datetime"] - df["prev_tx_time"]).dt.total_seconds() / 86400
    )

    median_gap = df["days_since_prev_tx"].median()
    df["days_since_prev_tx"] = df["days_since_prev_tx"].fillna(median_gap)

    df["inactive_long_gap"] = (df["days_since_prev_tx"] > 7).astype(int)

    df["inactive_big_inflow"] = (
        df["inactive_long_gap"] * df["is_inflow"] * df["is_high_value"]
    )


    # Large inflow → fast cash-out

    large_inflow_thresh = 20000
    df["large_inflow_flag"] = (
        (df["is_inflow"] == 1) &
        (df["Transaction_Amount"] >= large_inflow_thresh)
    ).astype(int)

    K = 3
    def recent_large(series):
        return series.shift(1).rolling(window=K, min_periods=1).max()

    df["recent_large_inflow"] = (
        df.groupby("Customer_ID")["large_inflow_flag"]
          .apply(recent_large)
          .reset_index(level=0, drop=True)
          .fillna(0)
    )

    df["cashout_after_large_inflow"] = (
        (df["recent_large_inflow"] == 1) & (df["is_outflow"] == 1)
    ).astype(int)


    # Interaction features

    df["highvalue_night"]      = df["is_high_value"] * df["is_night"]
    df["highvalue_weekend"]    = df["is_high_value"] * df["is_weekend"]
    df["ratio_night"]          = df["amt_to_balance_ratio"] * df["is_night"]
    df["is_business_acct"]     = (df["Account_Type"] == "Business").astype(int)
    df["busacct_highvalue"]    = df["is_business_acct"] * df["is_high_value"]

    df["hourbin_highvalue"]    = df["hour_bin"].astype(str) + "_" + df["is_high_value"].astype(str)
    df["txtype_night"]         = df["Transaction_Type"].astype(str) + "_" + df["is_night"].astype(str)


    # Cleanup

    df = df.drop(columns=[
        "Transaction_Date", "Transaction_Time", "prev_tx_time", "tx_datetime"
    ], errors="ignore")

    return df



# 2. Utility Functions

def keep_top_k(series, k=200, other_label="__OTHER__"):
    counts = Counter(series.dropna())
    keep = {v for v,_ in counts.most_common(k)}
    return series.where(series.isin(keep), other_label)

def imbalance_weights(y):
    pos = max((y == 1).sum(), 1)
    neg = max((y == 0).sum(), 1)
    w_pos = len(y) / (2 * pos)
    w_neg = len(y) / (2 * neg)
    return np.where(y == 1, w_pos, w_neg)

def tune_threshold_f1(y_true, scores):
    prec, rec, thr = precision_recall_curve(y_true, scores)
    f1 = [
        0 if p + r == 0 else 2 * p * r / (p + r)
        for p, r in zip(prec[:-1], rec[:-1])
    ]
    i = int(np.argmax(f1))
    return float(thr[i]), float(f1[i])


# 3. Load + FE
df = pd.read_csv("Bank_Transaction_Fraud_Detection.csv")
df = add_features(df)

print("Shape after FE:", df.shape)
print(df["Is_Fraud"].value_counts(normalize=True).rename("proportion"))

# Drop PII
df = df.drop(columns=[
    "Customer_Name","Customer_Contact","Customer_Email"
], errors="ignore")

# Reduce cardinality
for col in ["Merchant_ID","Customer_ID"]:
    df[col] = keep_top_k(df[col].astype(str), k=200)


# 4. Split

y = df["Is_Fraud"].astype(int)
X = df.drop(columns=["Is_Fraud"])

X_tr_full, X_te, y_tr_full, y_te = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_tr, X_va, y_tr, y_va = train_test_split(
    X_tr_full, y_tr_full, test_size=0.25, stratify=y_tr_full, random_state=42
)

# Categorical vs numeric
cat_cols = [c for c in X_tr.columns if X_tr[c].dtype == "object"]
num_cols = [c for c in X_tr.columns if c not in cat_cols]

print("Categorical:", cat_cols)
print("Numeric:", num_cols)

weights_tr = imbalance_weights(y_tr)


# 5. Preprocessor + NB Grid Search

def num_tf_template(n_bins):
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("bin", KBinsDiscretizer(
            n_bins=n_bins,
            encode="ordinal",
            strategy="quantile"
        ))
    ])

cat_tf = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ord", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

nonneg = FunctionTransformer(lambda X: np.clip(X + 1, 0, None))

bin_options   = [6, 8, 12]
alpha_options = [0.1, 0.5, 1.0, 2.0]

best_auc = -1
best_cfg = None
best_pipe = None

for nbins in bin_options:
    pre = ColumnTransformer([
        ("num", num_tf_template(nbins), num_cols),
        ("cat", cat_tf,               cat_cols)
    ])

    for alpha in alpha_options:
        pipe = Pipeline([
            ("pre", pre),
            ("nonneg", nonneg),
            ("nb", CategoricalNB(alpha=alpha))
        ])

        pipe.fit(X_tr, y_tr, nb__sample_weight=weights_tr)

        val_scores = pipe.predict_proba(X_va)[:, 1]
        auc = roc_auc_score(y_va, val_scores)

        if auc > best_auc:
            best_auc = auc
            best_cfg = {"n_bins": nbins, "alpha": alpha}
            best_pipe = pipe

print("\nBest config:", best_cfg)
print("Best Val AUC:", best_auc)


# 6. Threshold tuning

va_scores = best_pipe.predict_proba(X_va)[:, 1]

thr_f1, f1_val = tune_threshold_f1(y_va, va_scores)

print(f"\nBest F1 threshold: {thr_f1:.4f} | F1 = {f1_val:.4f}")


# 7. Test Evaluation

te_scores = best_pipe.predict_proba(X_te)[:, 1]

print("\nTest ROC-AUC:", roc_auc_score(y_te, te_scores))
print("Test PR-AUC :", average_precision_score(y_te, te_scores))

for name, thr in [("0.50", 0.50), ("F1", thr_f1)]:
    y_pred = (te_scores >= thr).astype(int)
    print(f"\n=== Threshold {name} ({thr:.4f}) ===")
    print(confusion_matrix(y_te, y_pred, labels=[1,0]))
    print(classification_report(y_te, y_pred, digits=4))


Shape after FE: (200000, 47)
Is_Fraud
0    0.94956
1    0.05044
Name: proportion, dtype: float64
Categorical: ['Customer_ID', 'Gender', 'State', 'City', 'Bank_Branch', 'Account_Type', 'Transaction_ID', 'Merchant_ID', 'Transaction_Type', 'Merchant_Category', 'Transaction_Device', 'Transaction_Location', 'Device_Type', 'Transaction_Currency', 'Transaction_Description', 'hour_bin', 'hourbin_highvalue', 'txtype_night']
Numeric: ['Age', 'Transaction_Amount', 'Account_Balance', 'tx_hour', 'tx_dayofweek', 'tx_month', 'is_weekend', 'is_night', 'amount_log', 'is_round_amount', 'is_high_value', 'amt_to_balance_ratio', 'is_inflow', 'is_outflow', 'days_since_prev_tx', 'inactive_long_gap', 'inactive_big_inflow', 'large_inflow_flag', 'recent_large_inflow', 'cashout_after_large_inflow', 'highvalue_night', 'highvalue_weekend', 'ratio_night', 'is_business_acct', 'busacct_highvalue']

Best config: {'n_bins': 8, 'alpha': 2.0}
Best Val AUC: 0.4925286683650754

Best F1 threshold: 0.2677 | F1 = 0.0966

Test